## Пробую для опенрутера

In [ ]:
from collections import namedtuple

from openai import OpenAI

In [32]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="YOUR TOKEN"
)

In [ ]:
def fetch_log_probs(text: str, model_name="qwen/qwen3-235b-a22b-2507", max_tokens=1):
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": text}],
        max_tokens=max_tokens,
        logprobs=True,
        top_logprobs=8,
        temperature=1,
        presence_penalty=2
    )
    return {prediction.token: prediction.logprob for prediction in response.choices[0].logprobs.content[0].top_logprobs}

In [ ]:
def generate_beams(initial_context: str, max_length=3, beam_width=8):
    BeamState = namedtuple('BeamState', ['current_text', 'token_path', 'score'])

    def expand_beam(beam_state):
        """Expand a single beam by getting top next tokens"""
        current_text, token_path, score = beam_state

        # Stop condition
        if len(token_path) >= max_length or any(punct in str(token_path[-1]) for punct in ['.', '!', '?', ' '] and len(token_path > 1)):
          print('Нашли сепаратор')
          return [beam_state]

        try:
            # print('Сейчас буду доставать токен для контекста: ', current_text)
            log_probs = fetch_log_probs(current_text)
            next_tokens = sorted(log_probs.items(), key=lambda x: x[1], reverse=True)[:beam_width]# лишняя сортировка, получаю уже сортированное

            new_beams = []
            for token, prob in next_tokens:
                new_text = f"{current_text}{token}"
                new_path = token_path + [token]
                new_score = score + prob  # accumulate probability
                new_beams.append(BeamState(new_text, new_path, new_score))

            return new_beams

        except Exception as e:
            print(f"Ошибка при генерации: {e}")
            return [beam_state]

    # Start with initial beam
    active_beams = [BeamState(initial_context, [], 0)]
    completed_beams = []

    # Expand beams until all are completed
    while active_beams:
        beam = active_beams.pop(0)
        expanded = expand_beam(beam)

        for new_beam in expanded:
            current_text, token_path, score = new_beam
            if (len(token_path) >= max_length or
                any(punct in str(token_path[-1]) for punct in ['.', '!', '?', ' '] and len(token_path) > 1)):
                completed_beams.append(new_beam)
            else:
                active_beams.append(new_beam)

    return completed_beams

In [ ]:
generate_beams(initial_context='Я уеду жить в ')

 ## Пробую на апи openai

In [1]:
import traceback

from openai import OpenAI

In [33]:
from openai import OpenAI

client = OpenAI(
    api_key="YOUR TOKEN"
)

In [19]:
def fetch_log_probs_openai(text: str, top_probs=8, max_tokens=1):
  '''
  Запрос к модели.
  '''
  response = client.completions.create(
        model="gpt-4o-mini",
        prompt= f'{text}',
        max_tokens=max_tokens,
        logprobs=top_probs,
        temperature=1,
    )
  return response.choices[0].logprobs.top_logprobs[0]

In [4]:
def dummy_fetch(text: str):
  return {'lol_one': 1, 'lol_two': 2, 'lol_three': 3}

In [31]:
def check_beam(token_path: list, max_length):
  '''
  Метод получает на вход список токенов луча и возвращает True/False в зависимости
  от того, нашелся ли в тексте токена символ сепаратора
  True  если нашелся сепаратор или длина луча (в токенах) превысила установленное значение в глубину
  '''
  if len(token_path) >= max_length or any(punct in str(token_path[-1]) for punct in {'.', '!', '?', ' ', '\n', '-', '"', ':', ';'}):
    return True
  return False


In [28]:
def generate_beams_openai(initial_context: str, max_length=3, beam_width=8):
    BeamState = namedtuple('BeamState', ['current_text', 'token_path', 'score_trace', 'finished'])

    def expand_beam(beam_state):
        """Expand a single beam by getting top next tokens"""
        current_text, token_path, score_trace, finished = beam_state

        # Stop condition
        if len(token_path) <= 1:
          pass
        elif check_beam(token_path, max_length):
          print('Нашли сепаратор')
          return [BeamState(current_text, token_path, score_trace, True)]

        try:
          # print('Сейчас буду доставать токен для контекста: ', current_text)
          log_probs = fetch_log_probs_openai(current_text)
          # log_probs = dummy_fetch(current_text)
          next_tokens = sorted(log_probs.items(), key=lambda x: x[1], reverse=True)[:beam_width]# лишняя сортировка, получаю уже сортированное
          new_beams = []

          for token, prob in next_tokens:
              new_text = f"{current_text}{token}"
              new_path = token_path + [token]
              new_score_trace = score_trace + [prob]  # accumulate probability
              new_beams.append(BeamState(new_text, new_path, new_score_trace, False))
          return new_beams

        except Exception as e:
          print(f"Ошибка при генерации: {e}")
          traceback.print_exc()
          return [beam_state]

    # Start with initial beam
    active_beams = [BeamState(initial_context, [], [], False)]
    completed_beams = []

    # Expand beams until all are completed
    while active_beams:
        beam = active_beams.pop(0)
        expanded = expand_beam(beam)

        for new_beam in expanded:
            current_text, token_path, score_trace, finished = new_beam
            if len(token_path) == 0:
              active_beams.append(new_beam)
            elif check_beam(token_path, max_length) or finished:
              completed_beams.append(new_beam)
            else:
              active_beams.append(new_beam)

    return completed_beams

In [10]:
res = generate_beams_openai(initial_context='Я уеду жить в ')

Нашли сепаратор
Нашли сепаратор
Нашли сепаратор
Нашли сепаратор


In [20]:
res

[BeamState(current_text='Я уеду жить в 905 год', token_path=['905', ' год'], score_trace=[-2.724236249923706, -2.086984634399414], finished=False),
 BeamState(current_text='Я уеду жить в 905.', token_path=['905', '.'], score_trace=[-2.724236249923706, -2.586984634399414], finished=False),
 BeamState(current_text='Я уеду жить в 905 году', token_path=['905', ' году'], score_trace=[-2.724236249923706, -3.211984634399414], finished=False),
 BeamState(current_text='Я уеду жить в 905.\n', token_path=['905', '.\n'], score_trace=[-2.724236249923706, -3.336984634399414], finished=False),
 BeamState(current_text='Я уеду жить в 10.', token_path=['10', '.'], score_trace=[-3.474236249923706, -2.418635368347168], finished=False),
 BeamState(current_text='Я уеду жить в 10 лет', token_path=['10', ' лет'], score_trace=[-3.474236249923706, -2.543635368347168], finished=False),
 BeamState(current_text='Я уеду жить в 10 стран', token_path=['10', ' стран'], score_trace=[-3.474236249923706, -2.6686353683471

In [29]:
res2_0 = generate_beams_openai(initial_context='Сегодня мы с друзьями идем в ')

In [30]:
res2_0

[BeamState(current_text='Сегодня мы с друзьями идем в 3-', token_path=['3', '-'], score_trace=[-2.2960517406463623, -4.598422527313232], finished=False),
 BeamState(current_text='Сегодня мы с друзьями идем в 3-D', token_path=['3', '-D'], score_trace=[-2.2960517406463623, -4.973422527313232], finished=False),
 BeamState(current_text='Сегодня мы с друзьями идем в 3 часа', token_path=['3', ' часа'], score_trace=[-2.2960517406463623, -5.098422527313232], finished=False),
 BeamState(current_text='Сегодня мы с друзьями идем в 3-х', token_path=['3', '-х'], score_trace=[-2.2960517406463623, -5.223422527313232], finished=False),
 BeamState(current_text='Сегодня мы с друзьями идем в 5-', token_path=['5', '-'], score_trace=[-2.6710517406463623, -2.764230251312256], finished=False),
 BeamState(current_text='Сегодня мы с друзьями идем в 5:', token_path=['5', ':'], score_trace=[-2.6710517406463623, -4.014230251312256], finished=False),
 BeamState(current_text='Сегодня мы с друзьями идем в 5-з', toke

In [23]:
res2_1 = generate_beams_openai(initial_context='Сегодня мы с друзьями идем в')

In [24]:
res2_1

[BeamState(current_text='Сегодня мы с друзьями идем в поход', token_path=[' поход'], score_trace=[-1.4680883884429932], finished=False),
 BeamState(current_text='Сегодня мы с друзьями идем в кино', token_path=[' кино'], score_trace=[-1.8430883884429932], finished=False),
 BeamState(current_text='Сегодня мы с друзьями идем в парк', token_path=[' парк'], score_trace=[-2.093088388442993], finished=False),
 BeamState(current_text='Сегодня мы с друзьями идем в кафе', token_path=[' кафе'], score_trace=[-2.718088388442993], finished=False),
 BeamState(current_text='Сегодня мы с друзьями идем в ресторан', token_path=[' ресторан'], score_trace=[-3.343088388442993], finished=False),
 BeamState(current_text='Сегодня мы с друзьями идем в гор', token_path=[' гор'], score_trace=[-3.468088388442993], finished=False),
 BeamState(current_text='Сегодня мы с друзьями идем в ак', token_path=[' ак'], score_trace=[-3.718088388442993], finished=False),
 BeamState(current_text='Сегодня мы с друзьями идем в те

In [14]:
res4 = generate_beams_openai(initial_context='Сегодня мы с друзьями ')

Нашли сепаратор
Нашли сепаратор


In [15]:
res3

[BeamState(current_text='Сегодня мы с друзьями  решили', token_path=[' решили'], score_trace=[-2.975886821746826], finished=False),
 BeamState(current_text='Сегодня мы с друзьями  по', token_path=[' по'], score_trace=[-3.475886821746826], finished=False),
 BeamState(current_text='Сегодня мы с друзьями 2 часа', token_path=['2', ' часа'], score_trace=[-3.350886821746826, -1.464900255203247], finished=False),
 BeamState(current_text='Сегодня мы с друзьями 2 дня', token_path=['2', ' дня'], score_trace=[-3.350886821746826, -2.089900255203247], finished=False),
 BeamState(current_text='Сегодня мы с друзьями 2 раза', token_path=['2', ' раза'], score_trace=[-3.350886821746826, -2.464900255203247], finished=False),
 BeamState(current_text='Сегодня мы с друзьями 2.', token_path=['2', '.'], score_trace=[-3.350886821746826, -2.964900255203247], finished=False),
 BeamState(current_text='Сегодня мы с друзьями 2 года', token_path=['2', ' года'], score_trace=[-3.350886821746826, -3.089900255203247], f